In [1]:
import optuna 
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pandas as pd


In [2]:
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
columns = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome']

In [3]:
df = pd.read_csv(url, names=columns)

In [4]:
df

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1
...,...,...,...,...,...,...,...,...,...
763,10,101,76,48,180,32.9,0.171,63,0
764,2,122,70,27,0,36.8,0.340,27,0
765,5,121,72,23,112,26.2,0.245,30,0
766,1,126,60,0,0,30.1,0.349,47,1


In [5]:
# handling missing values 
import numpy as np

df.isnull().sum()

Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64

In [6]:
(df==0).sum()


Pregnancies                 111
Glucose                       5
BloodPressure                35
SkinThickness               227
Insulin                     374
BMI                          11
DiabetesPedigreeFunction      0
Age                           0
Outcome                     500
dtype: int64

In [7]:
cols_with_missing_values = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI','DiabetesPedigreeFunction']
df[cols_with_missing_values] = df[cols_with_missing_values].replace(0, np.nan)
df

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148.0,72.0,35.0,NaN,33.6,0.627,50,1
1,1,85.0,66.0,29.0,NaN,26.6,0.351,31,0
2,8,183.0,64.0,NaN,NaN,23.3,0.672,32,1
3,1,89.0,66.0,23.0,94.0,28.1,0.167,21,0
4,0,137.0,40.0,35.0,168.0,43.1,2.288,33,1
...,...,...,...,...,...,...,...,...,...
763,10,101.0,76.0,48.0,180.0,32.9,0.171,63,0
764,2,122.0,70.0,27.0,NaN,36.8,0.340,27,0
765,5,121.0,72.0,23.0,112.0,26.2,0.245,30,0
766,1,126.0,60.0,NaN,NaN,30.1,0.349,47,1


In [8]:
df.fillna(df.median(), inplace=True)
df

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148.0,72.0,35.0,125.0,33.6,0.627,50,1
1,1,85.0,66.0,29.0,125.0,26.6,0.351,31,0
2,8,183.0,64.0,29.0,125.0,23.3,0.672,32,1
3,1,89.0,66.0,23.0,94.0,28.1,0.167,21,0
4,0,137.0,40.0,35.0,168.0,43.1,2.288,33,1
...,...,...,...,...,...,...,...,...,...
763,10,101.0,76.0,48.0,180.0,32.9,0.171,63,0
764,2,122.0,70.0,27.0,125.0,36.8,0.340,27,0
765,5,121.0,72.0,23.0,112.0,26.2,0.245,30,0
766,1,126.0,60.0,29.0,125.0,30.1,0.349,47,1


In [9]:
df.isnull().sum()

Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64

In [10]:
X = df.drop('Outcome', axis=1)
y = df['Outcome']

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.3, random_state=42)
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print(X_train.shape)
print(X_test.shape)

(537, 8)
(231, 8)


In [11]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

def objective(trial):
    # suggest values for the hyperparameters
    n_estimators = trial.suggest_int('n_estimators', 50, 200)
    max_depth = trial.suggest_int('max_depth', 3, 20)
    
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=42
    )
    
    score = cross_val_score(model, X_train, y_train, cv=5, scoring='accuracy').mean()
    return score


In [12]:
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler())
study.optimize(objective, n_trials=50)

[I 2026-06-29 14:35:08,653] A new study created in memory with name: no-name-da88bdba-ba70-4a56-8af9-d2270920e06f
[I 2026-06-29 14:35:08,981] Trial 0 finished with value: 0.7523018345448251 and parameters: {'n_estimators': 87, 'max_depth': 4}. Best is trial 0 with value: 0.7523018345448251.
[I 2026-06-29 14:35:09,559] Trial 1 finished with value: 0.7596919349255797 and parameters: {'n_estimators': 129, 'max_depth': 14}. Best is trial 1 with value: 0.7596919349255797.
[I 2026-06-29 14:35:10,083] Trial 2 finished with value: 0.7633610245759779 and parameters: {'n_estimators': 130, 'max_depth': 8}. Best is trial 2 with value: 0.7633610245759779.
[I 2026-06-29 14:35:10,353] Trial 3 finished with value: 0.7578746971270336 and parameters: {'n_estimators': 69, 'max_depth': 6}. Best is trial 2 with value: 0.7633610245759779.
[I 2026-06-29 14:35:10,664] Trial 4 finished with value: 0.76711664935964 and parameters: {'n_estimators': 73, 'max_depth': 16}. Best is trial 4 with value: 0.767116649359

In [13]:
# printing best results

print(f"best trial accuracy: {study.best_trial.value}")
print(f"best hyperparameters: {study.best_trial.params}")

best trial accuracy: 0.7690031152647976
best hyperparameters: {'n_estimators': 70, 'max_depth': 9}


In [14]:
from sklearn.metrics import accuracy_score

best_model = RandomForestClassifier(**study.best_trial.params, random_state=42)

best_model.fit(X_train, y_train)

y_pred = best_model.predict(X_test)

test_accuracy = accuracy_score(y_test, y_pred)

print(f"Test Accuracy with best hyperparameters: {test_accuracy:.2f}")

Test Accuracy with best hyperparameters: 0.75


In [15]:
from optuna.visualization import plot_optimization_history, plot_parallel_coordinate, plot_slice, plot_contour, plot_param_importances

In [16]:
plot_optimization_history(study).show()

In [17]:
plot_parallel_coordinate(study).show()

In [18]:
plot_slice(study).show()

In [20]:
plot_contour(study).show()

In [21]:
plot_param_importances(study).show()

In [22]:
# define by run

### optimize multiple ml models

In [23]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC


In [24]:
# Define the objective function for Optuna
def objective(trial):
    # Choose the algorithm to tune
    classifier_name = trial.suggest_categorical('classifier', ['SVM', 'RandomForest', 'GradientBoosting'])

    if classifier_name == 'SVM':
        # SVM hyperparameters
        c = trial.suggest_float('C', 0.1, 100, log=True)
        kernel = trial.suggest_categorical('kernel', ['linear', 'rbf', 'poly', 'sigmoid'])
        gamma = trial.suggest_categorical('gamma', ['scale', 'auto'])

        model = SVC(C=c, kernel=kernel, gamma=gamma, random_state=42)

    elif classifier_name == 'RandomForest':
        # Random Forest hyperparameters
        n_estimators = trial.suggest_int('n_estimators', 50, 300)
        max_depth = trial.suggest_int('max_depth', 3, 20)
        min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
        min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)
        bootstrap = trial.suggest_categorical('bootstrap', [True, False])

        model = RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            bootstrap=bootstrap,
            random_state=42
        )

    elif classifier_name == 'GradientBoosting':
        # Gradient Boosting hyperparameters
        n_estimators = trial.suggest_int('n_estimators', 50, 300)
        learning_rate = trial.suggest_float('learning_rate', 0.01, 0.3, log=True)
        max_depth = trial.suggest_int('max_depth', 3, 20)
        min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
        min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)

        model = GradientBoostingClassifier(
            n_estimators=n_estimators,
            learning_rate=learning_rate,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            random_state=42
        )

    # Perform cross-validation and return the mean accuracy
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy').mean()
    return score
        
        

In [25]:
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=100)

[I 2026-06-29 16:48:04,470] A new study created in memory with name: no-name-579c306a-cfc1-4fa1-8c44-135681feadea
[I 2026-06-29 16:48:04,491] Trial 0 finished with value: 0.7132216014897579 and parameters: {'classifier': 'SVM', 'C': 2.9756318845612784, 'kernel': 'sigmoid', 'gamma': 'scale'}. Best is trial 0 with value: 0.7132216014897579.
[I 2026-06-29 16:48:04,507] Trial 1 finished with value: 0.7355679702048418 and parameters: {'classifier': 'SVM', 'C': 0.7944117950370128, 'kernel': 'poly', 'gamma': 'scale'}. Best is trial 1 with value: 0.7355679702048418.
[I 2026-06-29 16:48:05,170] Trial 2 finished with value: 0.7635009310986964 and parameters: {'classifier': 'RandomForest', 'n_estimators': 282, 'max_depth': 13, 'min_samples_split': 5, 'min_samples_leaf': 8, 'bootstrap': True}. Best is trial 2 with value: 0.7635009310986964.
[I 2026-06-29 16:48:05,185] Trial 3 finished with value: 0.7690875232774674 and parameters: {'classifier': 'SVM', 'C': 0.5557543753940878, 'kernel': 'rbf', 'ga

In [26]:
study.trials_dataframe()

,number,value,datetime_start,datetime_complete,duration,params_C,params_bootstrap,params_classifier,params_gamma,params_kernel,params_learning_rate,params_max_depth,params_min_samples_leaf,params_min_samples_split,params_n_estimators,state
0,0,0.713222,2026-06-29 16:48:04.471412,2026-06-29 16:48:04.491819,0 days 00:00:00.020407,2.975632,NaN,SVM,scale,sigmoid,NaN,NaN,NaN,NaN,NaN,COMPLETE
1,1,0.735568,2026-06-29 16:48:04.492439,2026-06-29 16:48:04.507667,0 days 00:00:00.015228,0.794412,NaN,SVM,scale,poly,NaN,NaN,NaN,NaN,NaN,COMPLETE
2,2,0.763501,2026-06-29 16:48:04.508164,2026-06-29 16:48:05.170028,0 days 00:00:00.661864,NaN,True,RandomForest,NaN,NaN,NaN,13.0,8.0,5.0,282.0,COMPLETE
3,3,0.769088,2026-06-29 16:48:05.170553,2026-06-29 16:48:05.185530,0 days 00:00:00.014977,0.555754,NaN,SVM,auto,rbf,NaN,NaN,NaN,NaN,NaN,COMPLETE
4,4,0.763501,2026-06-29 16:48:05.186086,2026-06-29 16:48:05.319829,0 days 00:00:00.133743,NaN,False,RandomForest,NaN,NaN,NaN,18.0,1.0,8.0,53.0,COMPLETE
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,95,0.783985,2026-06-29 16:48:26.345950,2026-06-29 16:48:26.361429,0 days 00:00:00.015479,0.100299,NaN,SVM,scale,linear,NaN,NaN,NaN,NaN,NaN,COMPLETE
96,96,0.782123,2026-06-29 16:48:26.361941,2026-06-29 16:48:26.377260,0 days 00:00:00.015319,0.190726,NaN,SVM,scale,linear,NaN,NaN,NaN,NaN,NaN,COMPLETE
97,97,0.782123,2026-06-29 16:48:26.378036,2026-06-29 16:48:26.399577,0 days 00:00:00.021541,0.123420,NaN,SVM,scale,linear,NaN,NaN,NaN,NaN,NaN,COMPLETE
98,98,0.769088,2026-06-29 16:48:26.400549,2026-06-29 16:48:26.423867,0 days 00:00:00.023318,0.337371,NaN,SVM,scale,rbf,NaN,NaN,NaN,NaN,NaN,COMPLETE
